# cuda-gemm-lab on Colab

Runs the SGEMM kernel ladder: build, check every kernel against cuBLAS, benchmark, plot.

**First:** Runtime → Change runtime type → **T4 GPU** (or any GPU).

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/luciancj/cuda-gemm-lab.git"  # change if your fork lives elsewhere
!git clone -q {REPO_URL} 2>/dev/null || git -C cuda-gemm-lab pull -q
%cd cuda-gemm-lab

Compile for whichever GPU Colab gave us (T4 = `sm_75`, L4 = `sm_89`, A100 = `sm_80`).

In [ ]:
import subprocess
cc = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"], text=True
).split()[0]
ARCH = "sm_" + cc.replace(".", "")
print(ARCH)

In [ ]:
!make ARCH={ARCH} && make test ARCH={ARCH}

## Benchmark

The naive kernel at 4096³ takes a while; that is the point.

In [ ]:
!make bench ARCH={ARCH}

In [ ]:
!python scripts/plot.py results/bench.csv results/bench.png
from IPython.display import Image
Image("results/bench.png")

## Optional: profile one kernel with Nsight Compute

Colab sometimes blocks GPU performance counters (`ERR_NVGPUCTRPERM`); if so, skip this.
Compare `smem_tiling` (3) against `blocktile_2d` (5): look at *Memory Workload Analysis* and *Warp State Statistics*.

In [ ]:
!ncu --section SpeedOfLight --section MemoryWorkloadAnalysis -k regex:sgemm --launch-count 1 ./build/sgemm 5 2048